# FCCU Datensatz – Deep-Learning-basierte Rekonstruktionsmodelle für Novelty Detection

<div style="border-left: 4px solid #4F81BD; background:#F7FBFF; padding:10px 12px; margin:8px 0;">
<b>Lernziel</b><br>
Dieses Notebook zeigt rekonstruktionsbasierte Deep-Learning-Modelle für die frühe Fehlerdetektion im FCCU-Datensatz.
Trainiert wird ausschließlich auf gesundem Betrieb. Neue Daten werden danach darauf geprüft, ob sie noch zum Normalverhalten gehören.
</div>



In [ ]:
DATA_DIR = "./data/fccu"

FILES = {
    "NOC_stable": f"{DATA_DIR}/NOC_stableFeedFlow_outputs.csv",
    "NOC_varying": f"{DATA_DIR}/NOC_varyingFeedFlow_outputs.csv",
    "FAULT_sensorDrift": f"{DATA_DIR}/Fhn_sensorDrift_outputs.csv",
    "FAULT_UAf": f"{DATA_DIR}/UAf_decrease_outputs.csv",
   # "FAULT_condEff": f"{DATA_DIR}/condEff_decrease_outputs.csv",
    # "FAULT_deltaP": f"{DATA_DIR}/deltaP_increase_outputs.csv",
    # "FAULT_valveLeak": f"{DATA_DIR}/CAB_valveLeak_outputs.csv",
}

# Normalbetrieb
NOC_MODE = "stable"   # "stable", "varying" oder "both"; legt fest, welche NOC-Dateien genutzt werden

# Modelle gezielt aktivieren oder auskommentieren
ACTIVE_MODELS = [       # Liste der Modelle, die trainiert und verglichen werden sollen
    "LSTM-AE",
    "GRU-AE",
    "TCN",
    "TransformerEnc",
    "USAD",
    "OmniStyleVAE",
    "TranADLite",
]

# Sequenzierung
SEQ_LEN = 60            # Fensterlänge bzw. Sequenzlänge in Samples; bei 1 min Samplezeit = 60 Minuten
HOP = 10                # Schrittweite zwischen zwei Fenstern; bei 1 min Samplezeit = 10 Minuten

# Rekonstruktionsscore
#SCORE_MODE = "tail_mean"   # Standard: nur Mittelwert über die letzten TAIL_K Zeitschritte
SCORE_MODE = "full_mean" # Alternative: Mittelwert über das gesamte Fenster
#####
# bei tail_mean bitte unbedingt die Sequenzlänge / Fensterbreite beachten bei Eisntellung des Tails:
TAIL_K = 10                # Anzahl der letzten Zeitschritte, die bei tail_mean in den Score eingehen
#####

# Split vor der Sequenzbildung
NOC_TRAIN_FRAC = 0.50   # Anteil des Normalbetriebs für das Training
NOC_VAL_FRAC = 0.25     # Anteil des Normalbetriebs für Validierung und Schwellenkalibrierung
NOC_TEST_FRAC = 0.25    # Anteil des Normalbetriebs für den abschließenden Test

# Fault-Onset
SAMPLE_TIME_MINUTES = 1.0   # Abtastzeit in Minuten pro Zeitschritt
FAULT_ONSET = 120           # Onset des Fehlers bei 120 Minuten
FAULT_ONSET_INDEX = int(FAULT_ONSET / SAMPLE_TIME_MINUTES)   # Fehlerbeginn in Zeitschritt-Index umgerechnet
FAULT_ONSET_BY_KEY = {
    # "FAULT_sensorDrift": 120,   # optionaler faultspezifischer Fehlerbeginn in Minuten
}
FAULT_WINDOW_LABEL_RULE = "center"  # Regel, ab wann ein Fenster als fault-behaftet markiert wird bspw center oder end

# Training
RANDOM_STATE = 42        # Startwert für Zufallszahlen; sorgt für reproduzierbare Ergebnisse
TARGET_FAR = 0.001       # Zielwert für die False Alarm Rate bei der Schwellenkalibrierung
MAX_EPOCHS = 30          # maximale Anzahl an Trainingsdurchläufen pro Modell
PATIENCE = 5             # Anzahl an Epochen ohne Verbesserung bis zum Early Stopping
DEVICE = "cuda"         # Rechengerät: GPU mit CUDA oder alternativ CPU


# Alarmlogik
PERSISTENCE = 3          # Anzahl aufeinanderfolgender Überschreitungen bis ein Alarm als aktiv gilt

# Hyperparameter-Optimierung (HPO)
N_TRIALS = 6            # Anzahl der Versuche pro Modell in der Hyperparameter-Optimierung

BEST_PARAMS_JSON = "fccu_fdd_dl_optuna_best_params.json"   # Datei zum Speichern und Laden der optimierten Parameter
RUN_OPTUNA_STUDIES = False   # True: Hyperparameter-Optimierung neu starten
LOAD_BEST_PARAMS = True      # True: vorhandene JSON-Datei mit Best-Parametern laden
RUN_SECOND_RUN = True       # True: zweiten Run mit Bewertung und Grafiken ausführen

# Optionale Score-Glättung
USE_EWMA = False         # EWMA als optionale Glättung auf dem Rekonstruktionsscore
EWMA_ALPHA = 0.20        # Stärke der EWMA-Glättung

# CSV-Vorverarbeitung
TIME_COL_CANDIDATES = ["time", "timestamp", "t"]   # mögliche Namen der Zeitspalte
DROP_COLS = []           # Spalten, die vor der Modellierung entfernt werden sollen
USE_COLUMNS = None       # optionale explizite Spaltenauswahl
RESAMPLE_RULE = None     # optionale Resampling-Regel für Zeitdaten

print("Konfiguration geladen.")


## Datenlogik

Die NOC-Sequenzen werden zuerst chronologisch in Train, Validierung und Test geteilt. Erst danach werden Sequenzen gebaut.
Dadurch wird verhindert, dass stark überlappende Fenster zwischen Train und Validierung/Test vermischt werden.


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available() and DEVICE == "cuda":
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device:", device)

def _read_csv(path):
    df = pd.read_csv(path)
    for c in TIME_COL_CANDIDATES:
        if c in df.columns:
            try:
                df[c] = pd.to_datetime(df[c])
                df = df.sort_values(c).reset_index(drop=True)
                if RESAMPLE_RULE is not None:
                    df = df.set_index(c).resample(RESAMPLE_RULE).mean().interpolate().reset_index()
                break
            except Exception:
                pass
    return df

def _numeric(df):
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    if USE_COLUMNS is not None:
        use = [c for c in USE_COLUMNS if c in df.columns]
        num = df[use].select_dtypes(include=[np.number]).copy()
    else:
        num = df.select_dtypes(include=[np.number]).copy()
    num = num.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="any")
    return num

def load_files(files):
    out = {}
    for k, p in files.items():
        if not os.path.exists(p):
            print(f"⚠ fehlt: {k} -> {p}")
            continue
        df = _read_csv(p)
        out[k] = _numeric(df)
        print(f"✔ {k}: {df.shape} -> numerisch {out[k].shape}")
    return out

def align_common_columns(data_dict):
    keys = list(data_dict.keys())
    if not keys:
        return data_dict, []
    common = set(data_dict[keys[0]].columns)
    for k in keys[1:]:
        common &= set(data_dict[k].columns)
    common = [c for c in data_dict[keys[0]].columns if c in common]
    out = {k: data_dict[k][common].copy() for k in keys}
    return out, common

def split_sequence_chronological(X, train_frac, val_frac, test_frac):
    total = train_frac + val_frac + test_frac
    if not np.isclose(total, 1.0):
        raise ValueError("Split-Anteile müssen zusammen 1.0 ergeben.")
    T = len(X)
    i1 = int(np.floor(T * train_frac))
    i2 = int(np.floor(T * (train_frac + val_frac)))
    return X[:i1].copy(), X[i1:i2].copy(), X[i2:].copy()

def make_sequences_with_meta(X, seq_len, hop, source, split_name, onset_idx=None, label_rule="end"):
    T, D = X.shape
    if T < seq_len:
        meta = pd.DataFrame(columns=["source", "split", "start", "end", "center", "y", "post_onset", "onset_idx"])
        return np.empty((0, seq_len, D)), meta
    starts = list(range(0, T - seq_len + 1, hop))
    seqs = np.empty((len(starts), seq_len, D), dtype=float)
    rows = []
    for i, s in enumerate(starts):
        e = s + seq_len
        c = s + seq_len // 2
        seqs[i] = X[s:e, :]
        y = 0
        post_onset = False
        if onset_idx is not None:
            if label_rule == "start":
                post_onset = s >= onset_idx
            elif label_rule == "center":
                post_onset = c >= onset_idx
            else:
                post_onset = e > onset_idx
            y = int(post_onset)
        rows.append({"source": source, "split": split_name, "start": s, "end": e, "center": c, "y": y, "post_onset": int(post_onset), "onset_idx": onset_idx})
    meta = pd.DataFrame(rows)
    return seqs, meta

def combine_sequence_blocks(blocks):
    Xs, metas = [], []
    for Xb, mb in blocks:
        if len(Xb) == 0:
            continue
        Xs.append(Xb)
        metas.append(mb)
    if not Xs:
        return np.empty((0, 0, 0)), pd.DataFrame()
    X = np.concatenate(Xs, axis=0)
    meta = pd.concat(metas, axis=0, ignore_index=True)
    meta["global_idx"] = np.arange(len(meta))
    return X, meta

class SeqDataset(Dataset):
    def __init__(self, X):
        self.X = X.astype(np.float32)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i])

def fit_raw_scaler(sequence_map):
    mats = [v for v in sequence_map.values() if len(v) > 0]
    scaler = StandardScaler()
    scaler.fit(np.vstack(mats))
    return scaler

def transform_sequence_map(scaler, sequence_map):
    return {k: scaler.transform(v) if len(v) > 0 else v.copy() for k, v in sequence_map.items()}

def alarm_series(scores, thr, persistence=1):
    hits = np.asarray(scores) >= thr
    out = np.zeros(len(hits), dtype=bool)
    run = 0
    for i, h in enumerate(hits):
        run = run + 1 if h else 0
        if run >= persistence:
            out[i] = True
    return out

def first_true_index(x, start_idx=0):
    x = np.asarray(x, dtype=bool)
    if start_idx >= len(x):
        return -1
    hits = np.where(x[start_idx:])[0]
    return int(start_idx + hits[0]) if hits.size > 0 else -1


def eval_scores(y_true, scores, title=""):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    roc = np.nan
    ap = np.nan
    if len(np.unique(y_true)) > 1:
        roc = roc_auc_score(y_true, scores)
        ap = average_precision_score(y_true, scores)
    print(f"{title:<20} -> ROC-AUC={roc:.3f} | AP={ap:.3f}")
    return {"roc_auc": roc, "ap": ap}

def ewma_1d(x, alpha=0.2):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x.copy()
    out = np.empty_like(x)
    out[0] = x[0]
    for i in range(1, len(x)):
        out[i] = alpha * x[i] + (1.0 - alpha) * out[i - 1]
    return out

def ewma_by_source(scores, meta_df, alpha=0.2):
    scores = np.asarray(scores, dtype=float)
    out = np.empty_like(scores)
    for source, idx in meta_df.groupby("source").groups.items():
        idx = list(idx)
        out[idx] = ewma_1d(scores[idx], alpha=alpha)
    return out

print("Hilfsfunktionen bereit.")


In [ ]:
data = load_files(FILES)
if not data:
    raise RuntimeError("Keine Daten geladen. Pfade prüfen.")

data, feature_names = align_common_columns(data)
print("Gemeinsame numerische Spalten:", len(feature_names))

all_noc_keys = [k for k in data if k.startswith("NOC")]
fault_keys = [k for k in data if k.startswith("FAULT")]
assert len(all_noc_keys) >= 1 and len(fault_keys) >= 1, "Mindestens eine NOC- und eine Fault-Datei erforderlich."

if NOC_MODE == "stable":
    noc_keys = [k for k in all_noc_keys if "stable" in k]
elif NOC_MODE == "varying":
    noc_keys = [k for k in all_noc_keys if "varying" in k]
else:
    noc_keys = list(all_noc_keys)

print("Aktive NOC-Dateien:", noc_keys)
print("Aktive Fault-Dateien:", fault_keys)

raw_train_map = {}
raw_val_map = {}
raw_test_map = {}

for key in noc_keys:
    X = data[key].to_numpy(dtype=float)
    Xtr_raw, Xval_raw, Xte_raw = split_sequence_chronological(
        X,
        train_frac=NOC_TRAIN_FRAC,
        val_frac=NOC_VAL_FRAC,
        test_frac=NOC_TEST_FRAC,
    )
    raw_train_map[f"{key}__train"] = Xtr_raw
    raw_val_map[f"{key}__val"] = Xval_raw
    raw_test_map[f"{key}__test"] = Xte_raw

scaler = fit_raw_scaler(raw_train_map)
raw_train_map = transform_sequence_map(scaler, raw_train_map)
raw_val_map = transform_sequence_map(scaler, raw_val_map)
raw_test_map = transform_sequence_map(scaler, raw_test_map)
raw_fault_map = {k: scaler.transform(data[k].to_numpy(dtype=float)) for k in fault_keys}

train_blocks, val_blocks, test_blocks, fault_blocks = [], [], [], []
for key, X in raw_train_map.items():
    train_blocks.append(make_sequences_with_meta(X, SEQ_LEN, HOP, source=key, split_name="noc_train"))
for key, X in raw_val_map.items():
    val_blocks.append(make_sequences_with_meta(X, SEQ_LEN, HOP, source=key, split_name="noc_val"))
for key, X in raw_test_map.items():
    test_blocks.append(make_sequences_with_meta(X, SEQ_LEN, HOP, source=key, split_name="noc_test"))
for key, X in raw_fault_map.items():
    onset_idx = int(FAULT_ONSET_BY_KEY.get(key, FAULT_ONSET_INDEX))
    fault_blocks.append(
        make_sequences_with_meta(
            X,
            SEQ_LEN,
            HOP,
            source=key,
            split_name="fault_eval",
            onset_idx=onset_idx,
            label_rule=FAULT_WINDOW_LABEL_RULE,
        )
    )

X_train, meta_train = combine_sequence_blocks(train_blocks)
X_val, meta_val = combine_sequence_blocks(val_blocks)
X_noc_test, meta_noc_test = combine_sequence_blocks(test_blocks)
X_fault_eval, meta_fault_eval = combine_sequence_blocks(fault_blocks)

X_eval = np.concatenate([X_noc_test, X_fault_eval], axis=0)
meta_eval = pd.concat([meta_noc_test, meta_fault_eval], axis=0, ignore_index=True)
meta_eval["global_idx"] = np.arange(len(meta_eval))
y_eval = meta_eval["y"].to_numpy(dtype=int)

print("Sequenzen:")
print("  Train NOC:", X_train.shape)
print("  Val   NOC:", X_val.shape)
print("  Test  NOC:", X_noc_test.shape)
print("  Fault Eval:", X_fault_eval.shape)
print("  Eval gesamt:", X_eval.shape, "| positiver Anteil:", y_eval.mean())


## Verwendete Modellfamilien

- LSTM-AE: rekurrentes Autoencoder-Modell für sequentielle Rekonstruktion
- GRU-AE: kompaktere rekurrente Variante mit GRU-Zellen
- TCN: zeitlich-konvolutionale Rekonstruktion mit Dilatationen
- TransformerEnc: kompakter transformerbasierter Rekonstruktor
- USAD: rekonstruktionsbasierte Anomalieerkennung mit zwei Decodern
- OmniStyleVAE: probabilistische rekonstruktive Variante mit latentem Raum
- TranADLite: kompakte transformerbasierte Zwei-Phasen-Rekonstruktion

Die Modellsektion soll später zusätzlich um direkte Links zu Originalpublikationen und Repositories ergänzt werden.


In [ ]:
class LSTMAE(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        do = dropout if n_layers > 1 else 0.0
        self.encoder = nn.LSTM(d_in, hidden, num_layers=n_layers, batch_first=True, dropout=do)
        self.decoder = nn.LSTM(d_in, hidden, num_layers=n_layers, batch_first=True, dropout=do)
        self.out = nn.Linear(hidden, d_in)
    def forward(self, x):
        B, T, D = x.shape
        _, (h, c) = self.encoder(x)
        dec_in = torch.zeros(B, T, D, device=x.device, dtype=x.dtype)
        dec_out, _ = self.decoder(dec_in, (h, c))
        return self.out(dec_out)

class GRUAE(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.1):
        super().__init__()
        do = dropout if n_layers > 1 else 0.0
        self.encoder = nn.GRU(d_in, hidden, num_layers=n_layers, batch_first=True, dropout=do)
        self.decoder = nn.GRU(d_in, hidden, num_layers=n_layers, batch_first=True, dropout=do)
        self.out = nn.Linear(hidden, d_in)
    def forward(self, x):
        B, T, D = x.shape
        _, h = self.encoder(x)
        dec_in = torch.zeros(B, T, D, device=x.device, dtype=x.dtype)
        dec_out, _ = self.decoder(dec_in, h)
        return self.out(dec_out)

class Chomp1d(nn.Module):
    def __init__(self, chomp):
        super().__init__()
        self.chomp = chomp
    def forward(self, x):
        return x[:, :, :-self.chomp] if self.chomp > 0 else x

class TCNBlock(nn.Module):
    def __init__(self, d_in, d_out, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(d_in, d_out, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(d_out, d_out, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(d_in, d_out, 1) if d_in != d_out else nn.Identity()
    def forward(self, x):
        return self.net(x) + self.down(x)

class TCNReconstructor(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, kernel_size=3, dropout=0.1):
        super().__init__()
        layers = []
        ch_in = d_in
        for i in range(n_layers):
            layers.append(TCNBlock(ch_in, hidden, kernel_size=kernel_size, dilation=2 ** i, dropout=dropout))
            ch_in = hidden
        layers.append(nn.Conv1d(ch_in, d_in, 1))
        self.tcn = nn.Sequential(*layers)
    def forward(self, x):
        z = x.transpose(1, 2)
        y = self.tcn(z)
        return y.transpose(1, 2)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TransformerRecon(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(d_in, hidden)
        self.pos = PositionalEncoding(hidden)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out = nn.Linear(hidden, d_in)
    def forward(self, x):
        z = self.inp(x)
        z = self.pos(z)
        z = self.enc(z)
        return self.out(z)

class USAD(nn.Module):
    def __init__(self, seq_len, d_in, hidden=128, z_dim=64):
        super().__init__()
        flat_dim = seq_len * d_in
        self.encoder = nn.Sequential(
            nn.Linear(flat_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, z_dim), nn.ReLU(),
        )
        self.decoder1 = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, flat_dim),
        )
        self.decoder2 = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, flat_dim),
        )
        self.seq_len = seq_len
        self.d_in = d_in
    def forward(self, x):
        flat = x.reshape(x.size(0), -1)
        z = self.encoder(flat)
        w1 = self.decoder1(z).reshape(x.size(0), self.seq_len, self.d_in)
        w2 = self.decoder2(z).reshape(x.size(0), self.seq_len, self.d_in)
        z_w1 = self.encoder(w1.reshape(x.size(0), -1))
        w3 = self.decoder2(z_w1).reshape(x.size(0), self.seq_len, self.d_in)
        return w1, w2, w3

class OmniStyleVAE(nn.Module):
    def __init__(self, d_in, hidden=64, z_dim=32, n_layers=1, dropout=0.1):
        super().__init__()
        do = dropout if n_layers > 1 else 0.0
        self.enc = nn.GRU(d_in, hidden, num_layers=n_layers, batch_first=True, dropout=do)
        self.mu = nn.Linear(hidden, z_dim)
        self.logvar = nn.Linear(hidden, z_dim)
        self.z_to_h = nn.Linear(z_dim, hidden)
        self.dec = nn.GRU(d_in, hidden, num_layers=1, batch_first=True)
        self.out = nn.Linear(hidden, d_in)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self, x):
        B, T, D = x.shape
        _, h = self.enc(x)
        h_last = h[-1]
        mu = self.mu(h_last)
        logvar = self.logvar(h_last)
        z = self.reparameterize(mu, logvar)
        h0 = self.z_to_h(z).unsqueeze(0)
        dec_in = torch.zeros(B, T, D, device=x.device, dtype=x.dtype)
        dec_out, _ = self.dec(dec_in, h0)
        recon = self.out(dec_out)
        return recon, mu, logvar

class TranADLite(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.inp = nn.Linear(d_in, hidden)
        self.pos = PositionalEncoding(hidden)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=n_heads,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.enc1 = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.enc2 = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out = nn.Linear(hidden, d_in)
    def _block(self, x, enc):
        z = self.inp(x)
        z = self.pos(z)
        z = enc(z)
        return self.out(z)
    def forward(self, x):
        phase1 = self._block(x, self.enc1)
        resid = torch.abs(x - phase1)
        phase2 = self._block(x + resid, self.enc2)
        return phase1, phase2


In [ ]:
def reconstruction_score_per_sequence(x, recon, score_mode="tail_mean", tail_k=10):
    err_t = ((recon - x) ** 2).mean(dim=2)
    if score_mode == "full_mean":
        return err_t.mean(dim=1)
    tail_k = min(tail_k, err_t.size(1))
    tail = err_t[:, -tail_k:]
    return tail.mean(dim=1)

def build_model(name, d_in, params):
    if name == "LSTM-AE":
        return LSTMAE(d_in, params["hidden"], params["n_layers"], params["dropout"]), "recon"
    if name == "GRU-AE":
        return GRUAE(d_in, params["hidden"], params["n_layers"], params["dropout"]), "recon"
    if name == "TCN":
        return TCNReconstructor(d_in, params["hidden"], params["n_layers"], params["kernel_size"], params["dropout"]), "recon"
    if name == "TransformerEnc":
        return TransformerRecon(d_in, params["hidden"], params["n_layers"], params["n_heads"], params["dropout"]), "recon"
    if name == "USAD":
        return USAD(SEQ_LEN, d_in, params["hidden"], params["z_dim"]), "usad"
    if name == "OmniStyleVAE":
        return OmniStyleVAE(d_in, params["hidden"], params["z_dim"], params["n_layers"], params["dropout"]), "vae"
    if name == "TranADLite":
        return TranADLite(d_in, params["hidden"], params["n_layers"], params["n_heads"], params["dropout"]), "tranad"
    raise ValueError(name)

def train_model(model, mode, train_loader, val_loader, max_epochs=15, lr=1e-3, wd=1e-5, patience=5, beta=1e-3, verbose=True):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    best = float("inf")
    best_state = None
    bad = 0

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    for ep in range(1, max_epochs + 1):
        model.train()
        tr = 0.0
        for xb in train_loader:
            xb = xb.to(device)
            opt.zero_grad()

            if mode == "recon":
                recon = model(xb)
                loss = ((recon - xb) ** 2).mean()
            elif mode == "vae":
                recon, mu, logvar = model(xb)
                rec = ((recon - xb) ** 2).mean()
                kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = rec + beta * kld
            elif mode == "usad":
                w1, w2, w3 = model(xb)
                loss1 = 0.5 * ((xb - w1) ** 2).mean() + 0.5 * ((xb - w3) ** 2).mean()
                loss2 = 0.5 * ((xb - w2) ** 2).mean() - 0.5 * ((xb - w3) ** 2).mean()
                loss = loss1 + loss2
            elif mode == "tranad":
                p1, p2 = model(xb)
                loss = 0.5 * ((p1 - xb) ** 2).mean() + 0.5 * ((p2 - xb) ** 2).mean()
            else:
                raise ValueError(mode)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += loss.item() * len(xb)

        tr /= len(train_loader.dataset)

        model.eval()
        va = 0.0
        with torch.no_grad():
            for xb in val_loader:
                xb = xb.to(device)

                if mode == "recon":
                    recon = model(xb)
                    loss = ((recon - xb) ** 2).mean()
                elif mode == "vae":
                    recon, mu, logvar = model(xb)
                    rec = ((recon - xb) ** 2).mean()
                    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = rec + beta * kld
                elif mode == "usad":
                    w1, w2, w3 = model(xb)
                    loss1 = 0.5 * ((xb - w1) ** 2).mean() + 0.5 * ((xb - w3) ** 2).mean()
                    loss2 = 0.5 * ((xb - w2) ** 2).mean() - 0.5 * ((xb - w3) ** 2).mean()
                    loss = loss1 + loss2
                elif mode == "tranad":
                    p1, p2 = model(xb)
                    loss = 0.5 * ((p1 - xb) ** 2).mean() + 0.5 * ((p2 - xb) ** 2).mean()
                else:
                    raise ValueError(mode)

                va += loss.item() * len(xb)

        va /= len(val_loader.dataset)

        history["train_loss"].append(tr)
        history["val_loss"].append(va)

        if verbose:
            print(f"Epoch {ep:03d} | train={tr:.6f} | val={va:.6f}")

        if va < best - 1e-5:
            best = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                if verbose:
                    print(f"Early stopping nach Epoche {ep}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best, history
    
def score_model(model, mode, loader, score_mode="tail_mean", tail_k=10):
    model.eval(); model.to(device); scores = []
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            if mode == "recon":
                recon = model(xb); sc = reconstruction_score_per_sequence(xb, recon, score_mode=score_mode, tail_k=tail_k)
            elif mode == "vae":
                recon, mu, logvar = model(xb); rec = reconstruction_score_per_sequence(xb, recon, score_mode=score_mode, tail_k=tail_k); kld = 0.5 * torch.mean(mu.pow(2) + logvar.exp() - 1 - logvar, dim=1); sc = rec + 0.1 * kld
            elif mode == "usad":
                w1, w2, w3 = model(xb); e1 = reconstruction_score_per_sequence(xb, w1, score_mode=score_mode, tail_k=tail_k); e3 = reconstruction_score_per_sequence(xb, w3, score_mode=score_mode, tail_k=tail_k); sc = 0.5 * e1 + 0.5 * e3
            elif mode == "tranad":
                p1, p2 = model(xb); e1 = reconstruction_score_per_sequence(xb, p1, score_mode=score_mode, tail_k=tail_k); e2 = reconstruction_score_per_sequence(xb, p2, score_mode=score_mode, tail_k=tail_k); sc = 0.5 * e1 + 0.5 * e2
            scores.append(sc.detach().cpu().numpy())
    return np.concatenate(scores)

def plot_training_history(history, title="Trainingsverlauf"):
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.xlabel("Epoche")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Hyperparameter-Optimierung (HPO) mit Optuna

Optuna führt eine automatische Hyperparameter-Optimierung (HPO) durch. Ziel ist es, für jedes neuronale Netz geeignete Einstellungen zu finden,
zum Beispiel für versteckte Dimensionen, Lernrate, Schichtzahl, Dropout oder Batch-Größe.

In diesem Notebook wird die Suche bewusst nur auf unabhängigen Normaldaten durchgeführt.
Damit wird vermieden, dass Fault-Labels schon in der Hyperparameter-Optimierung die spätere Bewertung verzerren.


In [ ]:
#!pip install optuna

In [ ]:
try:
    import optuna
    OPTUNA_AVAILABLE = True
    print("Optuna erkannt.")
except Exception:
    OPTUNA_AVAILABLE = False
    print("Optuna nicht installiert. (pip install optuna)")

def suggest_params(trial, model_name):
    params = {
        "hidden": trial.suggest_categorical("hidden", [32, 64, 96, 128]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3, step=0.1),
        "lr": trial.suggest_categorical("lr", [5e-4, 1e-3, 2e-3]),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.0, 1e-6, 1e-5]),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }
    if model_name in ["LSTM-AE", "GRU-AE", "OmniStyleVAE"]:
        params["n_layers"] = trial.suggest_int("n_layers", 1, 3)
    if model_name == "TCN":
        params["n_layers"] = trial.suggest_int("n_layers", 1, 3)
        params["kernel_size"] = trial.suggest_categorical("kernel_size", [3, 5, 7])
    if model_name in ["TransformerEnc", "TranADLite"]:
        params["n_layers"] = trial.suggest_int("n_layers", 1, 3)
        params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
    if model_name in ["USAD", "OmniStyleVAE"]:
        params["z_dim"] = trial.suggest_categorical("z_dim", [16, 32, 64])
    return params

def normal_only_objective(trial, model_name):
    params = suggest_params(trial, model_name)
    d_in = X_train.shape[2]
    model, mode = build_model(model_name, d_in, params)
    train_loader = DataLoader(SeqDataset(X_train), batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(SeqDataset(X_val), batch_size=params["batch_size"], shuffle=False)
    model, best_val, history = train_model(model, mode, train_loader, val_loader, max_epochs=MAX_EPOCHS, lr=params["lr"], wd=params["weight_decay"], patience=PATIENCE, verbose=False)
    val_scores = score_model(model, mode, val_loader, score_mode=SCORE_MODE, tail_k=TAIL_K)
    med = float(np.median(val_scores)); iqr = float(np.quantile(val_scores, 0.75) - np.quantile(val_scores, 0.25)); q95 = float(np.quantile(val_scores, 0.95))
    return med + 0.5 * iqr + 0.5 * q95

def run_optuna(model_name, n_trials=8, study_name=None):
    if not OPTUNA_AVAILABLE:
        print("Optuna nicht verfügbar – überspringe."); return None
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="minimize", sampler=sampler, study_name=study_name)
    study.optimize(lambda tr: normal_only_objective(tr, model_name), n_trials=n_trials)
    print("Bestes Optuna-Ergebnis:", study.best_value)
    print("Beste Parameter:", study.best_params)
    return study

def save_best_params(best_dict, path="fccu_fdd_dl_optuna_best_params.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(best_dict, f, indent=2)
    print(f"✔ Best-Parameter gespeichert in: {path}")

def load_best_params(path="fccu_fdd_dl_optuna_best_params.json"):
    if not os.path.exists(path):
        print(f"⚠ Datei nicht gefunden: {path}"); return {}
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    display(pd.DataFrame.from_dict(d, orient="index").reset_index().rename(columns={"index": "model"}))
    return d

def run_all_studies_and_save(models=None, n_trials=N_TRIALS, out_json="fccu_fdd_dl_optuna_best_params.json"):
    if models is None:
        models = ACTIVE_MODELS
    if not OPTUNA_AVAILABLE:
        print("⚠ Optuna nicht verfügbar."); return {}
    best = {}
    for m in models:
        print(f"\n▶ Optuna für: {m}")
        study = run_optuna(m, n_trials=n_trials, study_name=f"FCCU_{m}")
        if study is not None:
            best[m] = study.best_params
    if best:
        save_best_params(best, out_json)
    return best


In [ ]:
best_params = load_best_params("fccu_fdd_dl_optuna_best_params.json")

if best_params:
    print("✔ Gespeicherte Best-Parameter werden verwendet.")
else:
    print("⚠ Keine gespeicherten Best-Parameter gefunden.")

## Bewertungsfunktionen

In dieser Zelle wird festgelegt, wie die Modelle bewertet werden.
Dazu gehören die FAR-basierte Schwelle, Alarmepisoden im gesunden Testbetrieb,
der First Alarm sowie die Detektionsverzögerung pro Fault relativ zum Fault-Onset.


In [ ]:
def collect_fault_segments(meta_df):
    segments = []
    for source, block in meta_df[meta_df["split"] == "fault_eval"].groupby("source", sort=False):
        block = block.sort_values("global_idx")
        onset_rows = block[block["post_onset"] == 1]
        onset_global = int(onset_rows["global_idx"].iloc[0]) if len(onset_rows) > 0 else None
        segments.append({
            "source": source,
            "start_global": int(block["global_idx"].min()),
            "end_global": int(block["global_idx"].max()) + 1,
            "onset_global": onset_global,
        })
    return segments
    
def evaluate_alarm_metrics(model_name, scores_val, scores_noc_test, scores_eval, meta_eval, persistence=3):
    thr = float(np.quantile(np.asarray(scores_val, dtype=float), 1.0 - TARGET_FAR))
    noc_alarm = alarm_series(scores_noc_test, thr, persistence=persistence)
    noc_alarm_episodes = int(noc_alarm[0]) + int(np.sum((noc_alarm[1:] == 1) & (noc_alarm[:-1] == 0))) if len(noc_alarm) > 0 else 0
    noc_hours = (len(scores_noc_test) * HOP * SAMPLE_TIME_MINUTES) / 60.0
    false_alarms_per_hour = noc_alarm_episodes / noc_hours if noc_hours > 0 else np.nan
    alarm_share = float(np.mean(noc_alarm)) if len(noc_alarm) > 0 else np.nan

    fault_segments = collect_fault_segments(meta_eval)
    full_alarm = alarm_series(scores_eval, thr, persistence=persistence)

    per_fault = []
    for seg in fault_segments:
        start_idx = seg["onset_global"] if seg["onset_global"] is not None else seg["start_global"]
        hits = np.where(full_alarm[start_idx:])[0]
        first_alarm = int(start_idx + hits[0]) if hits.size > 0 else -1
        if first_alarm >= seg["end_global"]:
            first_alarm = -1
        delay = (first_alarm - start_idx) if first_alarm >= 0 else np.nan
        per_fault.append({
            "Modell": model_name,
            "Fault": seg["source"],
            "Onset idx": start_idx,
            "First alarm idx": first_alarm if first_alarm >= 0 else np.nan,
            "Detektionsverzögerung (Fenster)": delay,
            "Detektionsverzögerung (min)": delay * HOP * SAMPLE_TIME_MINUTES if np.isfinite(delay) else np.nan,
        })

    valid_delays = [r["Detektionsverzögerung (Fenster)"] for r in per_fault if np.isfinite(r["Detektionsverzögerung (Fenster)"])]
    mean_delay = float(np.mean(valid_delays)) if valid_delays else np.nan

    first_fault_onset = min(
        [seg["onset_global"] for seg in fault_segments if seg["onset_global"] is not None],
        default=None
    )

    start_global = first_fault_onset if first_fault_onset is not None else len(scores_noc_test)

    # Alarme vor dem Fault-Onset werden für den globalen First Alarm ignoriert
    alarm_after_onset = full_alarm.copy()
    alarm_after_onset[:start_global] = False

    hits_global = np.where(alarm_after_onset)[0]
    global_first_alarm = int(hits_global[0]) if hits_global.size > 0 else -1

    summary = {
        "Modell": model_name,
        "Schwelle": thr,
        "False alarms / h (NOC-Test)": false_alarms_per_hour,
        "Alarmanteil (NOC-Test)": alarm_share,
        "Alarmepisoden (NOC-Test)": noc_alarm_episodes,
        "First alarm idx": global_first_alarm if global_first_alarm >= 0 else np.nan,
        "Detektionsverzögerung (Fenster)": mean_delay,
        "Detektionsverzögerung (min)": mean_delay * HOP * SAMPLE_TIME_MINUTES if np.isfinite(mean_delay) else np.nan,
    }
    return summary, pd.DataFrame(per_fault), thr

def plot_single_model_score(scores_eval, meta_eval, thr, model_name, persistence=3):
    alarm = alarm_series(scores_eval, thr, persistence=persistence)
    xs = np.arange(len(scores_eval))
    fig = plt.figure(figsize=(12, 4))
    ax = plt.gca()
    ax.plot(xs, scores_eval, label=f"{model_name} – Rekonstruktionsscore")
    ax.axhline(thr, color="C1", linestyle="--", alpha=0.9, label=f"Schwelle @ FAR={TARGET_FAR}")

    y0, y1 = np.nanmin(scores_eval), np.nanmax(scores_eval)
    yr = max(y1 - y0, 1e-9)

    for source, block in meta_eval.groupby("source", sort=False):
        s0 = int(block["global_idx"].min())
        ax.axvline(s0, linestyle=":", alpha=0.4, color="gray")
        ax.text(s0, y0 + 0.04 * yr, str(source), rotation=90, va="bottom", ha="left", fontsize=8, color="gray")
        onset_rows = block[block["post_onset"] == 1]
        if len(onset_rows) > 0:
            onset_idx = int(onset_rows["global_idx"].iloc[0])
            ax.axvline(onset_idx, color="C3", linestyle="--", alpha=0.7)
            ax.text(onset_idx, y0 + 0.92 * yr, "Fault Onset", rotation=90, va="top", ha="left", fontsize=8, color="C3")

    fault_segments = collect_fault_segments(meta_eval)
    first_fault_onset = min(
        [seg["onset_global"] for seg in fault_segments if seg["onset_global"] is not None],
        default=None
    )
    start_global = first_fault_onset if first_fault_onset is not None else len(scores_eval)

    alarm_after_onset = alarm.copy()
    alarm_after_onset[:start_global] = False

    hits = np.where(alarm_after_onset)[0]
    first_alarm = int(hits[0]) if hits.size > 0 else -1

    if first_alarm >= 0:
        ax.axvline(first_alarm, color="C2", linestyle="-", alpha=0.8, label="Erster persistenter Alarm")

    ax.set_xlabel("Sequenzindex")
    ax.set_ylabel("Rekonstruktionsscore")
    ax.set_title(f"{model_name}: Score, Schwelle, Fault Onset und Alarm")
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_scores_with_fault_segments(scores_dict, meta_eval, thresholds=None, title_prefix="", zoom_fault_only=False):
    if not scores_dict:
        print("⚠ Keine Scores für Plot")
        return

    fig = plt.figure(figsize=(12, 4))
    ax = plt.gca()
    max_len = max(len(v) for v in scores_dict.values())
    x0 = 0
    if zoom_fault_only:
        fault_only = meta_eval[meta_eval["split"] == "fault_eval"]
        if len(fault_only) > 0:
            x0 = int(fault_only["global_idx"].min())

    xs = np.arange(max_len)
    for name, sc in scores_dict.items():
        xx = xs[:len(sc)]
        ax.plot(xx, sc, label=name)
        if thresholds is not None and name in thresholds and np.isfinite(thresholds[name]):
            ax.axhline(thresholds[name], linestyle="--", alpha=0.35)

    y0, y1 = ax.get_ylim()
    for source, block in meta_eval.groupby("source", sort=False):
        s0 = int(block["global_idx"].min())
        ax.axvline(s0, linestyle=":", alpha=0.5)
        ax.text(s0, y0 + 0.05 * (y1 - y0), str(source), rotation=90, va="bottom", ha="left")
        onset_rows = block[block["post_onset"] == 1]
        if len(onset_rows) > 0:
            og = int(onset_rows["global_idx"].iloc[0])
            ax.axvline(og, linestyle="--", alpha=0.7, color="C3")

    ax.set_xlim(x0, max_len)
    ax.set_xlabel("Sequenzindex")
    ax.set_ylabel("Rekonstruktionsscore")
    ax.set_title((title_prefix + " – " if title_prefix else "") + ("Fault-Zoom" if zoom_fault_only else "Gesamt"))
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_summary_bars(df_results):
    if df_results is None or len(df_results) == 0:
        print("⚠ Keine Ergebnisse für Balkendiagramme vorhanden.")
        return

    order = df_results["Modell"].tolist()
    x = np.arange(len(order))

    # Detektionsverzögerung
    vals = df_results["Detektionsverzögerung (Fenster)"].to_numpy(dtype=float)
    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    bars = ax.bar(x, np.nan_to_num(vals, nan=0.0))
    for i, v in enumerate(vals):
        if np.isnan(v):
            bars[i].set_alpha(0.3)
            bars[i].set_hatch("//")
            ax.text(i, 0.01, "NaN", ha="center", va="bottom", fontsize=9)
        else:
            ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylabel("Detektionsverzögerung (Fenster)")
    ax.set_title("Detektionsverzögerung pro Modell")
    plt.tight_layout()
    plt.show()

    # First Alarm
    vals2 = df_results["First alarm idx"].to_numpy(dtype=float)
    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    bars2 = ax.bar(x, np.nan_to_num(vals2, nan=0.0))
    for i, v in enumerate(vals2):
        if np.isnan(v):
            bars2[i].set_alpha(0.3)
            bars2[i].set_hatch("//")
            ax.text(i, 0.01, "NaN", ha="center", va="bottom", fontsize=9)
        else:
            ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylabel("First Alarm Index")
    ax.set_title("First Alarm pro Modell")
    plt.tight_layout()
    plt.show()


## Zweiter Run mit den besten Parametern

Im ersten Schritt wurden mit Optuna gute Hyperparameter auf Normaldaten gesucht.
In diesem zweiten Run werden diese gespeicherten Parameter geladen und die Modelle damit erneut trainiert und vollständig bewertet.

So entsteht die eigentliche Vergleichsbasis zwischen den Modellen mit festen, bereits optimierten Einstellungen.


In [ ]:
def second_run_all_faults(best_params=None, best_params_path=BEST_PARAMS_JSON, models=None):
    if best_params is None:
        best = load_best_params(best_params_path)
    else:
        best = best_params
    if not best:
        print("⚠ Keine Best-Parameter gefunden.")
        return None

    if models is None:
        models = tuple(best.keys())

    d_in = X_train.shape[2]
    results = []
    per_fault_tables = []
    scores_all = {}
    thresholds = {}

    for m in models:
        params = best[m]
        bs = int(params.get("batch_size", 64))
        train_loader = DataLoader(SeqDataset(X_train), batch_size=bs, shuffle=True)
        val_loader = DataLoader(SeqDataset(X_val), batch_size=bs, shuffle=False)
        noc_test_loader = DataLoader(SeqDataset(X_noc_test), batch_size=bs, shuffle=False)
        eval_loader = DataLoader(SeqDataset(X_eval), batch_size=bs, shuffle=False)

        model, mode = build_model(m, d_in, params)
        print(f"\n=== Training Modell: {m} ===")

        model, best_val_loss, history = train_model(
            model,
            mode,
            train_loader,
            val_loader,
            max_epochs=MAX_EPOCHS,
            lr=params.get("lr", 1e-3),
            wd=params.get("weight_decay", 0.0),
            patience=PATIENCE,
        )
        plot_training_history(history, title=f"{m} – Trainingsverlauf")
        
        scores_val = score_model(model, mode, val_loader, score_mode=SCORE_MODE, tail_k=TAIL_K)
        scores_noc_test = score_model(model, mode, noc_test_loader, score_mode=SCORE_MODE, tail_k=TAIL_K)
        scores_eval = score_model(model, mode, eval_loader, score_mode=SCORE_MODE, tail_k=TAIL_K)

        # Klassifikationsmetriken auf Eval-Daten
        cls_metrics = eval_scores(y_eval, scores_eval, m)

        # Alarmmetriken und Schwellenkalibrierung
        alarm_summary, fault_df, thr = evaluate_alarm_metrics(
            m,
            scores_val=scores_val,
            scores_noc_test=scores_noc_test,
            scores_eval=scores_eval,
            meta_eval=meta_eval,
            persistence=PERSISTENCE,
        )
        
        results.append({**cls_metrics, **alarm_summary, "best_val_loss": best_val_loss})
        per_fault_tables.append(fault_df)
        scores_all[m] = scores_eval
        thresholds[m] = thr

        plot_single_model_score(scores_eval, meta_eval, thr, m, persistence=PERSISTENCE)

    df_results = pd.DataFrame(results).sort_values("Modell").reset_index(drop=True)
    df_faults = pd.concat(per_fault_tables, axis=0, ignore_index=True) if per_fault_tables else pd.DataFrame()

    display(df_results)
    if len(df_faults) > 0:
        display(df_faults)

    df_results.to_csv("second_run_all_faults_metrics.csv", index=False)
    if len(df_faults) > 0:
        df_faults.to_csv("second_run_all_faults_per_fault.csv", index=False)

    plot_scores_with_fault_segments(scores_all, meta_eval, thresholds=thresholds, title_prefix="All Faults", zoom_fault_only=False)
    plot_scores_with_fault_segments(scores_all, meta_eval, thresholds=thresholds, title_prefix="All Faults", zoom_fault_only=True)
    plot_summary_bars(df_results)

    return df_results, df_faults, scores_all, thresholds


## Ausführungszellen

Ab hier werden die Modelle tatsächlich ausgeführt.

Die Steuerung erfolgt jetzt über die Schalter in der Config:
- `RUN_OPTUNA_STUDIES = True` startet die Hyperparameter-Optimierung neu
- `LOAD_BEST_PARAMS = True` lädt vorhandene Best-Parameter aus der JSON-Datei
- `RUN_SECOND_RUN = True` startet den vollständigen Modellvergleich mit Tabellen und Grafiken

Damit wird verhindert, dass Optuna automatisch doppelt startet oder JSON-Dateien mehrfach geladen werden.


In [ ]:
best_params = {}

if RUN_OPTUNA_STUDIES and LOAD_BEST_PARAMS:
    print("⚠ Sowohl RUN_OPTUNA_STUDIES als auch LOAD_BEST_PARAMS sind aktiv.")
    print("   Es wird zuerst Optuna ausgeführt und danach die neu gespeicherte JSON-Datei verwendet.")

if RUN_OPTUNA_STUDIES:
    best_params = run_all_studies_and_save(
        models=ACTIVE_MODELS,
        n_trials=N_TRIALS,
        out_json=BEST_PARAMS_JSON,
    )
    if best_params:
        df_best_params = (
            pd.DataFrame.from_dict(best_params, orient="index")
            .reset_index()
            .rename(columns={"index": "model"})
        )
        display(df_best_params)
    else:
        print("⚠ Keine Best-Parameter aus Optuna verfügbar.")

if LOAD_BEST_PARAMS:
    best_params = load_best_params(BEST_PARAMS_JSON)
    if best_params:
        print("✔ Gespeicherte Best-Parameter werden verwendet.")
    else:
        print("⚠ Keine gespeicherten Best-Parameter gefunden.")

if not RUN_OPTUNA_STUDIES and not LOAD_BEST_PARAMS:
    print("ℹ Weder Optuna noch JSON-Laden aktiviert. Es wurden noch keine Best-Parameter bereitgestellt.")


## Zweiter Run

Der zweite Run wird nur gestartet, wenn `RUN_SECOND_RUN = True` gesetzt ist.
Dabei werden die aktuell verfügbaren Best-Parameter verwendet, entweder aus der neuen Optuna-Studie oder aus der geladenen JSON-Datei.


In [ ]:
if RUN_SECOND_RUN:
    df_results, df_faults, scores_all, thresholds = second_run_all_faults(
        best_params=best_params if best_params else None,
        best_params_path=BEST_PARAMS_JSON,
        models=ACTIVE_MODELS,
    )

    display(
        df_results[[
            "Modell",
            "roc_auc",
            "ap",
            "False alarms / h (NOC-Test)",
            "Detektionsverzögerung (Fenster)",
            "Detektionsverzögerung (min)",
        ]].sort_values("Detektionsverzögerung (Fenster)")
    )
else:
    print("ℹ RUN_SECOND_RUN = False. Der vollständige Modellvergleich wurde nicht gestartet.")
